# OpenAI text-embedding-3-large Baseline — Summary-Only Variant
## Commercial API dense retrieval — 3072-dimensional embeddings

**What this notebook does:**

Implements the OpenAI `text-embedding-3-large` dense retrieval baseline using
*only* the free-text `summary` field — no structured fields (name, country,
industry code, keywords). This is the "Set A" baseline: how well does this
commercial model do with just the description text alone, as a comparison
point against `04_baseline_openai_large.ipynb`'s all-fields variant.

**Why text-embedding-3-large and not 3-small?**
- `text-embedding-3-large` produces **3072-dimensional** embeddings vs 1536-d for small
- MTEB score of **64.6** vs 62.3 for small — meaningfully stronger
- Cost is ~$0.13/M tokens — encoding full corpus costs approximately **$1.95 total**
- Given the rich text format with structured metadata appears to challenge open-source
  models, a larger commercial model trained on more diverse data may handle it better

**Important differences from local models:**
- Encoding happens on OpenAI's servers — no local GPU needed for encoding
- Query latency includes network round-trip (~200ms) which is not a model property
- Latency is broken into: network time + local FAISS search for fair comparison
- Uses personal OpenAI API key (NOT the university Mannheim proxy)

**FAISS index type:** `IndexFlatIP`
OpenAI embeddings are unit-normalised by default — use inner product (cosine similarity).

**Folder structure:**
```
result/
└── 04_baseline_openai_large_summary/
    ├── company_embeddings.npy        # OpenAI 3072-d embeddings (98716 x 3072)
    ├── company_faiss.index           # FAISS IndexFlatIP index
    ├── openai_large_results.csv      # Top-1000 results per query (101 queries)
    ├── latency_breakdown.csv         # Network vs search latency per query
    └── evaluation_openai_large.csv   # NDCG, Precision, Recall, F1 @ k in {10,50,100,1000}
```

### Notebook structure
1. Environment setup & cost estimate
2. Imports
3. Load dataset & build corpus (summary field only)
4. Encode all companies → embeddings (batched API calls)
5. Build FAISS index
6. Run all 101 queries + measure latency breakdown
7. Evaluation — NDCG, Precision, Recall, F1
8. Final summary

## 1 · Environment Setup & Cost Estimate

**Cost calculation before running:**
- Model: `text-embedding-3-large` at $0.13 per million tokens
- Corpus: ~98,716 companies × ~150 tokens average per rich text = ~15M tokens
- Estimated cost: ~**$1.95** for full corpus encoding (one-time)
- Query encoding: 101 queries × ~10 tokens = negligible (~$0.0001)

This uses your **personal OpenAI API key** — NOT the university Mannheim proxy.
The `.env` file should contain `OPENAI_API_KEY=sk-...`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True: .env always wins over any stray shell-exported OPENAI_API_KEY
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    raise ValueError('[Setup] OPENAI_API_KEY not found in .env — add it before running')

RESULT_DIR = Path('result/04_baseline_openai_large_summary')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder  : {RESULT_DIR}/ — ready')
print(f'[Setup] API key found  : sk-...{OPENAI_API_KEY[-6:]}')
print(f'[Setup] Model          : text-embedding-3-large')
print(f'[Setup] Embedding dims : 3072')
print(f'[Setup] Est. cost      : ~$1.95 for full corpus encoding')

## 2 · Imports

| Package | Role |
|---|---|
| `openai` | Official OpenAI Python client — calls the embeddings API |
| `faiss` | Facebook AI Similarity Search — fast vector nearest-neighbour lookup |
| `pandas / numpy` | Data loading and numerical operations |
| `time` | Measuring encoding and query latency |
| `tqdm` | Progress bar for batch encoding |

In [ ]:
from openai import OpenAI
import faiss
import json, time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Initialise OpenAI client — personal key, no base_url
client = OpenAI(api_key=OPENAI_API_KEY)

RESULT_DIR = Path('result/04_baseline_openai_large_summary')
MODEL_NAME = 'text-embedding-3-large'
EMBED_DIMS = 3072

# Quick connectivity test — encode one string to confirm API key works
print('[Test] Testing API connection...')
t0   = time.time()
test = client.embeddings.create(input=['test'], model=MODEL_NAME)
print(f'[Test] API connected in  : {(time.time()-t0)*1000:.0f}ms')
print(f'[Test] Embedding dims    : {len(test.data[0].embedding)}')
print(f'[Test] Model confirmed   : {MODEL_NAME}')
print('[Imports] All packages loaded successfully')

[Test] Testing API connection...
[Test] API connected in  : 1437ms
[Test] Embedding dims    : 3072
[Test] Model confirmed   : text-embedding-3-large
[Imports] All packages loaded successfully


## 3 · Load Dataset & Build Corpus

**Why summary-only for this variant?**
This is the "Set A" baseline: embed *only* the free-text `summary` field, with
no structured fields (name, country, industry code, keywords) to fall back on.
This isolates how much retrieval quality this model gets from the summary text
alone, as a comparison point against `04_baseline_openai_large.ipynb`'s
all-fields variant.

In [ ]:
print('[Load] Loading production results...')
results_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Total rows        : {len(results_df):,}')
print(f'[Load] Columns available : {list(results_df.columns)}')

all_companies = results_df.drop_duplicates(subset='domain').reset_index(drop=True)
print(f'[Load] Unique companies  : {len(all_companies):,}')

with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries           : {len(data)}')

# ── Build rich text — identical to BM25, MiniLM, BGE notebooks ───────────────
print('[Load] Building rich text for each company...')

def build_rich_text(row):
    """Summary only — Set A baseline experiments."""
    return str(row.get('summary', '')) if pd.notna(row.get('summary')) else ''

rich_texts = [build_rich_text(row) for _, row in all_companies.iterrows()]

# ── Token count estimate ──────────────────────────────────────────────────────
avg_chars    = sum(len(t) for t in rich_texts) / len(rich_texts)
avg_tokens   = avg_chars / 4  # rough estimate: 1 token ≈ 4 chars
total_tokens = avg_tokens * len(rich_texts)
est_cost     = total_tokens / 1_000_000 * 0.13

print(f'[Load] Avg text length   : {avg_chars:.0f} chars (~{avg_tokens:.0f} tokens)')
print(f'[Load] Total tokens est. : {total_tokens/1e6:.1f}M tokens')
print(f'[Load] Estimated cost    : ${est_cost:.2f}')
print(f'[Load] Sample rich text  :')
print(f'  {rich_texts[0][:300]}...')

[Load] Loading production results...
[Load] Total rows        : 101,000
[Load] Columns available : ['rank', 'similarity_score', 'domain', 'name', 'organization_type', 'organization_size', 'country', 'state', 'district', 'municipality', 'summary', 'summary_keywords', 'nace_code', 'query_id', 'query']
[Load] Unique companies  : 98,716
[Load] Queries           : 101
[Load] Building rich text for each company...
[Load] Avg text length   : 804 chars (~201 tokens)
[Load] Total tokens est. : 19.8M tokens
[Load] Estimated cost    : $2.58
[Load] Sample rich text  :
  Company: Software Genesis, Inc. | Country: United States | State: Illinois | Type: Company | Size: Micro (0-9) | Industry: NACE K: Telecommunication, computer programming, consulting, computing infrastructure and other information service activities | Software Genesis is a software development compa...


## 4 · Encode All Companies via OpenAI API

Encoding is done in batches of 500 texts per API call — the maximum allowed.
Each batch is a single HTTP request to OpenAI's servers.

**Why batching matters:**
- 98,716 companies / 500 per batch = ~198 API calls
- Each call returns 500 embeddings of 3072 dimensions
- Batching reduces total network overhead significantly vs one call per company

**Embeddings are saved immediately** after encoding completes — do not
re-encode unnecessarily as it costs money.

**OpenAI embeddings are unit-normalised by default** — we use `IndexFlatIP`.

In [ ]:
BATCH_SIZE = 500

# ── Check if embeddings already exist — skip encoding if they do ──────────────
embeddings_path = RESULT_DIR / 'company_embeddings.npy'

if embeddings_path.exists():
    print(f'[Encode] Embeddings already exist — loading from disk...')
    print(f'[Encode] Skipping API encoding — no cost incurred')
    embeddings  = np.load(embeddings_path).astype('float32')
    ENCODE_TIME = 0.0
    print(f'[Encode] Loaded shape  : {embeddings.shape}')
    print(f'[Encode] Dims          : {embeddings.shape[1]}')

else:
    print(f'[Encode] No saved embeddings found — encoding via API...')
    print(f'[Encode] Estimated cost: ~$1.95')

    def encode_batch(texts, model=MODEL_NAME):
        texts    = [t if t.strip() else 'unknown company' for t in texts]
        response = client.embeddings.create(input=texts, model=model)
        sorted_data = sorted(response.data, key=lambda x: x.index)
        return [item.embedding for item in sorted_data]

    all_embeddings = []
    total_start    = time.time()
    n_batches      = (len(rich_texts) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in tqdm(range(0, len(rich_texts), BATCH_SIZE),
                  total=n_batches, desc='Encoding'):
        batch      = rich_texts[i:i + BATCH_SIZE]
        batch_embs = encode_batch(batch)
        all_embeddings.extend(batch_embs)

        batch_num = i // BATCH_SIZE + 1
        if batch_num % 20 == 0:
            elapsed   = time.time() - total_start
            done      = len(all_embeddings)
            remaining = (len(rich_texts) - done) * elapsed / done
            print(f'[Encode] {done:,}/{len(rich_texts):,} encoded  |  ~{remaining:.0f}s remaining')

    ENCODE_TIME = time.time() - total_start
    embeddings  = np.array(all_embeddings, dtype='float32')

    print(f'[Encode] Done in          : {ENCODE_TIME/60:.1f} minutes')
    print(f'[Encode] Embeddings shape : {embeddings.shape}')

    norms = np.linalg.norm(embeddings[:5], axis=1)
    print(f'[Encode] Sample norms     : {norms.tolist()}')

    np.save(embeddings_path, embeddings)
    print(f'[Encode] Saved to         : {embeddings_path}')

## 5 · Build FAISS Index

**Index type: `IndexFlatIP`** (Inner Product = Cosine Similarity)

OpenAI embeddings are unit-normalised by default, so inner product equals
cosine similarity. Same index type as BGE — different from MiniLM which uses `IndexFlatL2`.

The index is built from the locally saved `.npy` file — no API call needed.

In [ ]:
print('[FAISS] Loading embeddings...')
embeddings = np.load(RESULT_DIR / 'company_embeddings.npy').astype('float32')
print(f'[FAISS] Embeddings shape  : {embeddings.shape}')

print('[FAISS] Building IndexFlatIP...')
t0        = time.time()
dimension = embeddings.shape[1]  # 3072
index     = faiss.IndexFlatIP(dimension)
index.add(embeddings)
INDEX_BUILD_TIME = time.time() - t0

print(f'[FAISS] Index built in    : {INDEX_BUILD_TIME:.2f}s')
print(f'[FAISS] Vectors in index  : {index.ntotal:,}')
print(f'[FAISS] Index type        : IndexFlatIP (cosine similarity)')
print(f'[FAISS] Vector dims       : {dimension}')

faiss.write_index(index, str(RESULT_DIR / 'company_faiss.index'))
print(f'[FAISS] Index saved to    : {RESULT_DIR}/company_faiss.index')

## 6 · Run All 101 Queries + Measure Latency Breakdown

**Query pipeline:**
1. Send query string to OpenAI API → get 3072-d embedding (network call)
2. Search local FAISS index for top-1000 nearest companies (local computation)

**Latency breakdown (critical for fair comparison):**
- `network_ms` = time for OpenAI API to return the query embedding
- `search_ms` = time for local FAISS search
- `total_ms` = network + search

The network component is NOT a model property — it depends on internet speed.
For a fair comparison with local models (BGE, MiniLM), compare only `search_ms`.

**Output:** `result/04_baseline_openai_large_summary/openai_large_results.csv`

In [ ]:
print(f'[Run] Starting retrieval for {len(data)} queries...')
print(f'[Run] Retrieving top-1000 per query')
print(f'[Run] Each query = 1 API call (network) + 1 FAISS search (local)')
print('-' * 60)

all_results   = []
latency_rows  = []
query_times   = []
total_start   = time.time()

for i, item in enumerate(data):
    query_id = item['query_id']
    query    = item['query']

    # ── Step 1: Encode query via API — measure network time ───────────────────
    t0         = time.perf_counter()
    response   = client.embeddings.create(input=[query], model=MODEL_NAME)
    q_emb      = np.array([response.data[0].embedding], dtype='float32')
    network_ms = (time.perf_counter() - t0) * 1000

    # ── Step 2: FAISS search — measure local search time ─────────────────────
    t0         = time.perf_counter()
    scores, idxs = index.search(q_emb, 1000)
    search_ms  = (time.perf_counter() - t0) * 1000

    total_ms   = network_ms + search_ms
    query_times.append(total_ms)

    # ── Collect results ───────────────────────────────────────────────────────
    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0])):
        company = all_companies.iloc[idx]
        all_results.append({
            'query_id': query_id,
            'query':    query,
            'rank':     rank + 1,
            'score':    float(score),
            'domain':   company['domain'],
            'name':     company.get('name', ''),
            'country':  company.get('country', ''),
            'summary':  company.get('summary', ''),
        })

    latency_rows.append({
        'query_id':   query_id,
        'query':      query,
        'network_ms': round(network_ms, 1),
        'search_ms':  round(search_ms, 1),
        'total_ms':   round(total_ms, 1),
    })

    if (i + 1) % 20 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        remaining = (len(data) - i - 1) * elapsed / (i + 1)
        avg_net   = sum(r['network_ms'] for r in latency_rows) / len(latency_rows)
        avg_srch  = sum(r['search_ms']  for r in latency_rows) / len(latency_rows)
        print(f'[Run] {i+1:3d}/{len(data)}  |  '
              f'avg network={avg_net:.0f}ms  search={avg_srch:.0f}ms  |  '
              f'~{remaining:.0f}s remaining')

# ── Save results ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
latency_df = pd.DataFrame(latency_rows)

results_df.to_csv(RESULT_DIR / 'openai_large_results.csv', index=False)
latency_df.to_csv(RESULT_DIR / 'latency_breakdown.csv',    index=False)

AVG_NETWORK_MS = latency_df['network_ms'].mean()
AVG_SEARCH_MS  = latency_df['search_ms'].mean()
AVG_TOTAL_MS   = latency_df['total_ms'].mean()

print('-' * 60)
print(f'[Run] Done!')
print(f'[Run] Total results         : {len(results_df):,}')
print(f'[Run] Avg network latency   : {AVG_NETWORK_MS:.1f}ms  (API round-trip)')
print(f'[Run] Avg search latency    : {AVG_SEARCH_MS:.1f}ms   (local FAISS)')
print(f'[Run] Avg total latency     : {AVG_TOTAL_MS:.1f}ms')
print(f'[Run] Results saved to      : {RESULT_DIR}/openai_large_results.csv')
print(f'[Run] Latency saved to      : {RESULT_DIR}/latency_breakdown.csv')

## 7 · Evaluation — NDCG, Precision, Recall, F1 @ k

Same evaluation protocol as all other baseline notebooks.

### Pseudo-relevance labels
A company is **relevant** for a query if it appears in the **production top-100**.

### Metrics at k ∈ {10, 50, 100, 1000}

| Metric | What it measures |
|---|---|
| **NDCG@k** | Ranking quality — rewards relevant results ranked higher |
| **Precision@k** | Of top-k results, what fraction are relevant? |
| **Recall@k** | Of all relevant companies, what fraction did we find? |
| **F1@k** | Harmonic mean of Precision and Recall |

In [ ]:
print('[Eval] Loading production labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_VALUES = [10, 50, 100, 500, 1000]

def get_relevant(query_id, top_k=1000):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(
        1 / np.log2(i + 2)
        for i, d in enumerate(retrieved[:k]) if d in relevant
    )

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

print('[Eval] Computing metrics for all queries...')
eval_rows = []

for i, item in enumerate(data):
    qid       = item['query_id']
    query     = item['query']
    relevant  = get_relevant(qid)
    retrieved = (
        results_df[results_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval_rows.append({
            'query_id':  qid,
            'query':     query,
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval] {i+1}/101 queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_openai_large.csv', index=False)
print(f'[Eval] Saved to {RESULT_DIR}/evaluation_openai_large.csv')

## 8 · Final Summary

Full results averaged across all 101 queries.

**Latency note for thesis:**
Report `search_ms` for the fair comparison against local models.
Report `total_ms` as the real-world production latency.
The difference is purely network latency — not a model property.

In [ ]:
print('[Summary] ============================================================')
print('[Summary] OpenAI text-embedding-3-large RESULTS')
print('[Summary] ============================================================')
print(f'\n[Summary] Corpus encoding time  : {ENCODE_TIME/60:.1f} minutes')
print(f'[Summary] Index build time      : {INDEX_BUILD_TIME:.2f}s')
print(f'[Summary] Avg network latency   : {AVG_NETWORK_MS:.1f}ms  (API — not model speed)')
print(f'[Summary] Avg FAISS search      : {AVG_SEARCH_MS:.1f}ms   (local)')
print(f'[Summary] Avg total latency     : {AVG_TOTAL_MS:.1f}ms')
print(f'[Summary] Companies encoded     : {len(all_companies):,}')
print(f'[Summary] Embedding dims        : {EMBED_DIMS}')
print(f'[Summary] FAISS index type      : IndexFlatIP')
print(f'[Summary] Normalisation         : True (OpenAI returns unit vectors)')
print(f'[Summary] Queries evaluated     : {len(data)}')
print()
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval_df[eval_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')
print()
print(f'[Summary] Latency breakdown (for thesis):')
print(f'  Network (API round-trip) : {AVG_NETWORK_MS:.1f}ms  <- not comparable to local models')
print(f'  FAISS search (local)     : {AVG_SEARCH_MS:.1f}ms  <- fair comparison with BGE/MiniLM')
print('[Summary] ============================================================')